# PUF-Chain NDS — Figure Reproduction Notebook

This notebook regenerates **every publication figure used in `main.tex`** directly
from the seed-locked experiment pipeline in this repository, and asserts that the
values drawn on each figure match the canonical metrics in `output/results.json`
(which are the numbers `main.tex` reports).

**Pipeline**

1. `simulate.py` → `dataset/*.csv` + offline characterization
2. `run_experiments_v2.py` → adaptive ECC, XGBoost, Isolation-Forest metrics
3. `run_experiments_v11.py` → trimmed-mean fusion, Arrhenius, PBFT
4. `generate_paper_figures.py` → renders all 8 PDFs into `output/figs/`

**Figures produced** (same filenames `main.tex` includes):
`fig_cross_puf.pdf`, `fig_environment.pdf`, `fig_environment_2.pdf`,
`fig_per_bit_heatmap.pdf`, `fig_xgb_calibration.pdf`, `fig_adaptive_ecc.pdf`,
`fig_iso_forest.pdf`, `fig_robust_aggregation.pdf`.

Everything is deterministic at `seed = 42`. Run all cells top to bottom.

In [ ]:
# --- 0. Environment -----------------------------------------------------------
import sys, subprocess
# Figure rendering needs matplotlib + pandas (numpy is already required by the repo).
for pkg in ("numpy", "pandas", "matplotlib"):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("dependencies ready")

In [ ]:
# --- 1. Locate the repository root -------------------------------------------
from pathlib import Path
import json

ROOT = Path.cwd()
if not (ROOT / "simulate.py").exists():
    # allow running from a subdirectory
    for parent in ROOT.parents:
        if (parent / "simulate.py").exists():
            ROOT = parent
            break
if not (ROOT / "simulate.py").exists():
    raise FileNotFoundError("Run this notebook from the repository root that contains simulate.py")

sys.path.insert(0, str(ROOT))
DATASET_DIR  = ROOT / "dataset"
RESULTS_PATH = ROOT / "output" / "results.json"
FIG_DIR      = ROOT / "output" / "figs"
print("Repository root:", ROOT)

In [ ]:
# --- 2. Run the full experiment pipeline (regenerates dataset + results.json) -
import importlib
import simulate, run_experiments_v2, run_experiments_v11
for m in (simulate, run_experiments_v2, run_experiments_v11):
    importlib.reload(m)

simulate.main()
run_experiments_v2.main()
run_experiments_v11.main()
print("\nPipeline complete.")

In [ ]:
# --- 3. Load the canonical, paper-aligned metrics ----------------------------
results = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))

oc  = results["offline_characterization"]
v2  = results["v2"]
v11 = results["v11"]

print("PUF characterization (intra-HD %, inter-HD %)")
for fam in ("sram", "arbiter", "hybrid"):
    print(f"  {fam:8s} {oc[fam]['intra_hd_pct']:.2f}  {oc[fam]['inter_hd_pct']:.2f}")

print("\nAdaptive ECC :", v2["adaptive_ecc"]["cells"], "cells",
      f"({v2['adaptive_ecc']['saving_vs_512_pct']}% saving),",
      f"key recovery {v2['adaptive_ecc']['adaptive_key_recovery']*100:.2f}%")
print("Static rep-8 per regime:",
      {k: round(v*100, 2) for k, v in v2["regime_rep8_key_recovery"].items()})
print("Isolation Forest :", f"recall {v2['isolation_forest']['recall']*100:.2f}%,",
      f"FAR {v2['isolation_forest']['false_alarm_rate']*100:.2f}%,",
      f"AUC {v2['isolation_forest']['roc_auc']}")
print("XGBoost :", f"R2 {v2['xgboost_aggregate']['r2']}, MAE {v2['xgboost_aggregate']['mae_bits']} bits")
print("Trimmed mean :", v11["trimmed_mean_attack_bit_mae"]["fedavg"], "->",
      v11["trimmed_mean_attack_bit_mae"]["trimmed_mean"],
      f"({v11['trimmed_mean_attack_bit_mae']['improvement_ratio']:.1f}x)")

## 4. Render all figures

`generate_paper_figures.generate_all()` reads `output/results.json` for every
annotated number and `dataset/traffic_sram.csv` for the data-shape panels, then
writes each figure as both `.pdf` (for `main.tex`) and `.png` (shown below).

In [ ]:
# --- 4. Generate the figures --------------------------------------------------
import generate_paper_figures as gpf
importlib.reload(gpf)
paths = gpf.generate_all()
for name, p in paths.items():
    print("wrote", p.relative_to(ROOT), "(+ .png)")

In [ ]:
# --- 5. Display the figures inline -------------------------------------------
from IPython.display import Image, display, Markdown
order = [
    ("fig_cross_puf.png",         "Fig. PUF quality — intra/inter Hamming distance per family"),
    ("fig_environment.png",       "Fig. Environment — temperature dependence + aging"),
    ("fig_per_bit_heatmap.png",   "Fig. Per-bit flip-rate heatmap (drives adaptive ECC)"),
    ("fig_xgb_calibration.png",   "Fig. XGBoost calibration (R2=0.363, MAE=1.262)"),
    ("fig_adaptive_ecc.png",      "Fig. Adaptive vs static ECC — footprint + per-regime recovery"),
    ("fig_iso_forest.png",        "Fig. Isolation-Forest detection — scores + ROC"),
    ("fig_robust_aggregation.png","Fig. Byzantine-robust trimmed-mean fusion"),
]
for fname, caption in order:
    display(Markdown(f"**{caption}**"))
    display(Image(filename=str(FIG_DIR / fname)))

## 5. Verify every figure matches `main.tex`

The assertions below re-derive each figure's headline statistics from
`output/results.json` (and the repetition-code math in `fuzzy_extractor.py`)
and confirm they equal the values `main.tex` reports.

In [ ]:
# --- 6. Paper-alignment assertions -------------------------------------------
from math import comb

def bit_success(p, n):
    return sum(comb(n, k) * p**k * (1 - p)**(n - k) for k in range(0, (n - 1)//2 + 1))

def close(a, b, tol, label):
    assert abs(a - b) <= tol, f"{label}: got {a}, expected {b} (tol {tol})"
    print(f"  OK  {label}: {a}")

print("PUF quality (Table II / fig_cross_puf):")
close(oc["sram"]["intra_hd_pct"],   3.20, 0.01, "SRAM intra-HD")
close(oc["sram"]["inter_hd_pct"],  46.89, 0.01, "SRAM inter-HD")
close(oc["arbiter"]["intra_hd_pct"], 0.47, 0.01, "Arbiter intra-HD")
close(oc["hybrid"]["inter_hd_pct"], 50.30, 0.01, "Hybrid inter-HD")

print("\nAdaptive ECC (Table IV / fig_adaptive_ecc a):")
ae = v2["adaptive_ecc"]
assert ae["cells"] == 420 and ae["max_repetition"] == 9
close(ae["saving_vs_512_pct"], 18.0, 0.05, "cell saving %")
close(ae["adaptive_key_recovery"]*100, 99.64, 0.01, "adaptive recovery %")
close(ae["static_rep8_key_recovery_nominal"]*100, 99.24, 0.01, "static nominal %")

print("\nPer-regime static rep-8 recovery (Table III / fig_adaptive_ecc b):")
reg = v2["regime_rep8_key_recovery"]
for k, exp in {"cold": 99.55, "nominal": 99.68, "warm": 98.30, "attack": 68.29}.items():
    close(reg[k]*100, exp, 0.01, f"static {k} %")

print("\nAdaptive holds the 99% target in every regime (fig_adaptive_ecc b):")
for k, p in {"cold": 0.0325, "nominal": 0.0299, "warm": 0.0459, "attack": 0.1047}.items():
    tau_bit = 1 - (1 - 0.99) / 64
    n = next(N for N in range(1, 33, 2) if bit_success(p, N) >= tau_bit)
    rec = bit_success(p, n)**64
    assert rec >= 0.99, f"adaptive {k} below target"
    print(f"  OK  adaptive {k}: {rec*100:.2f}% (>= 99% target, N={n})")

print("\nXGBoost calibration (fig_xgb_calibration):")
close(v2["xgboost_aggregate"]["r2"], 0.363, 0.001, "R2")
close(v2["xgboost_aggregate"]["mae_bits"], 1.262, 0.001, "MAE bits")

print("\nIsolation Forest (fig_iso_forest):")
isf = v2["isolation_forest"]
close(isf["recall"]*100, 95.74, 0.01, "recall %")
close(isf["false_alarm_rate"]*100, 5.70, 0.01, "false-alarm %")
close(isf["roc_auc"], 0.9939, 1e-4, "ROC-AUC")

print("\nTrimmed-mean fusion (fig_robust_aggregation):")
tm = v11["trimmed_mean_attack_bit_mae"]
close(tm["fedavg"], 0.0414, 1e-4, "FedAvg attack-bit MAE")
close(tm["trimmed_mean"], 0.0022, 1e-4, "Trimmed attack-bit MAE")
close(tm["improvement_ratio"], 18.818, 0.01, "improvement ratio")

print("\nAll figure values match main.tex.")